# Test set and Train set extraction

The notebook takes care of extracting a test set from the cleaned combined dataframes.  

The test set contains around 12 job ads per month, with descriptions longer than 200 characters.

The test set ads are extracted from the train set to avoid data leakage and both new datasets are saved.

In [1]:
import pandas as pd
import json

with open("../data/clean_ads.json", encoding="utf-8") as f:
    clean_ads_df = pd.DataFrame(json.load(f))

print("Number of rows:", len(clean_ads_df))

Number of rows: 2729


In [2]:
# checking ads with short descriptions
shorty = clean_ads_df[clean_ads_df["description"].str.len() < 200]

print("Number of short descriptions:", len(shorty))
shorty.head()

Number of short descriptions: 3


,id,title,employer,municipality,region,occupation,published,description
0,30523189,PLM Teamcenter,ITMOTIVE in Sweden Aktiebolag,NaN,NaN,"Verksamhetskonsult, IT",2026-01-27T01:04:50,"PLM Teamcenter Developer, Solution Architect. ..."
1621,30999468,Category Analyst,Cool Company Skandinavien AB,Stockholm,Stockholms län,Systemanalytiker/Systemutredare,2026-05-06T08:32:20,Must have +5 experience.
2546,31417403,Senior SAP Consultant,Acumant AB,Solna,Stockholms län,"Verksamhetskonsult, IT",2026-08-31T12:26:50,Experience: e.g. 7–10+ years of SAP experience...


In [3]:
# removing jobs with short desc
restricted_df = clean_ads_df[clean_ads_df["description"].str.len() >= 200]

print("Number of rows after removal of short ones", len(restricted_df))

Number of rows after removal of short ones 2726


In [4]:
# length buckets
restricted_df["len_bucket"] = pd.qcut(restricted_df["description"].str.len(), 3, labels=["short", "medium", "long"])

# extracting month
restricted_df["published"] = pd.to_datetime(restricted_df["published"])
restricted_df["month"] = restricted_df["published"].dt.month

# shuffling
restricted_df = restricted_df.sample(frac=1, random_state=42)

In [5]:
# help function to select samples in a balanced way by length for test set
def sample_balanced(d, n_per_bucket=4, seed=42):
    return d.groupby(["month", "len_bucket"], observed=True).sample(n=n_per_bucket, random_state=seed)

In [6]:
# creating test set
test_set = pd.concat([sample_balanced(restricted_df)])

print(f"Number of ads in the test set: {len(test_set)}\n")
print("Number of ads selected per month and length bucket:\n")
table = pd.crosstab(test_set["month"], test_set["len_bucket"], margins=True, margins_name="Total")
print(table)

Number of ads in the test set: 120

Number of ads selected per month and length bucket:

len_bucket  short  medium  long  Total
month                                 
1               4       4     4     12
2               4       4     4     12
3               4       4     4     12
4               4       4     4     12
5               4       4     4     12
6               4       4     4     12
7               4       4     4     12
8               4       4     4     12
9               4       4     4     12
10              4       4     4     12
Total          40      40    40    120


In [14]:
# The test set only need "id" and "description", so I drop the rest of the cols
test_set = test_set[["id", "description"]].reset_index(drop=True)
test_set = test_set.rename(columns={"description": "text"})
test_set.head()

,id,text
0,30462188,Application hosting is looking for a Domain Ar...
1,30472655,Professional Galaxy is an IT and technology co...
2,30472352,Om företaget Avaron AB är ett växande konsultb...
3,30475382,Om företaget Avaron AB är ett växande konsultb...
4,30533854,Vill du arbeta med avancerad planering i en gl...


In [8]:
train_set = restricted_df[~restricted_df["id"].isin(test_set["id"])].reset_index(drop=True)

print("Number of rows in the restricted df:", len(restricted_df))
print("Number of rows in the test_set:", len(test_set))
print("Number of rows in the train_set:", len(train_set))
train_set.head()

Number of rows in the restricted df: 2726
Number of rows in the test_set: 120
Number of rows in the train_set: 2606


,id,title,employer,municipality,region,occupation,published,description,len_bucket,month
0,31266388,Chefsarkitekt – samla ihop och utveckla vår en...,REGERINGSKANSLIET,Stockholm,Stockholms län,Enterprise Architect,2026-07-10 15:15:55,"Digitaliseringsavdelningen, Sektionen för Stra...",long,7
1,30760800,Senior Solutions Architect SAP,Cepheid AB,Solna,Stockholms län,Systemarkitekt,2026-03-18 00:06:15,"At Cepheid, one of Danaher’s 15+ operating com...",medium,3
2,30770777,Ingenjör med inriktning mot tillgänglighetsanalys,Försvarets Materielverk,Stockholm,Stockholms län,Systemanalytiker/Systemutredare,2026-03-19 13:11:24,Har du erfarenhet av analysarbete och komplexa...,long,3
3,30623050,Lead AWS Engineer (Global Edge Infrastructure ...,Justera Group AB,Stockholm,Stockholms län,"Affärskonsult, IT",2026-02-17 12:00:11,About Justera Group Justera Group AB is a lead...,short,2
4,30558951,Service Delivery Manager IT,Bae Systems Hägglunds AB,Karlskoga,Örebro län,Enterprise Architect,2026-02-03 14:21:35,Vill du ta en ledande roll i att forma vår fra...,long,2


In [9]:
# Dropping unused cols and converting timestamp back to string to be able to save in .json
train_set = train_set.drop(columns=["month", "len_bucket"], errors="ignore")
train_set["published"] = pd.to_datetime(train_set["published"]).dt.strftime("%Y-%m-%dT%H:%M:%S")

In [15]:
# Saving sets
with open("../data/test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_set.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

with open("../data/train_set.json", "w", encoding="utf-8") as f:
    json.dump(train_set.to_dict(orient="records"), f, ensure_ascii=False, indent=2)